# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/DianaMayalo/ML-internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

`Lane: Refresh / Content Opportunity Scoring. I am choosing this lane because it turns a scoring/ranking problem something I already have experience with from prior predictive modeling and dashboard work into a decision-support tool a content team could actually act on. The starter pipeline already demonstrates this lane end-to-end, which gives me a working baseline to critique and improve rather than starting from a blank question.`

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

`For a content strategist with limited weekly review capacity, deciding which pages to prioritize for refresh, we will build a ranked scoring queue from the FlyRank content/search dataset, scoring refresh priority measured by precision@50 (and average precision). A wrong call costs wasted reviewer time (false positive) or a missed, silently-declining high-value page (false negative, more costly). A plain rule isn't enough because the underlying signals (visibility, freshness, position, engagement) interact and shift over time in ways a single if-statement can't capture across hundreds of thousands of pages. We will claim only observed and decision-support results — not causal proof that a refresh improves outcomes.`

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import pandas as pd

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

# --- Gotcha check first: confirm scale of rate columns ---
print(df[["ctr", "engagement_rate", "trend_pct"]].describe())
# ctr/engagement_rate should look like small numbers (e.g. 0-5 range), not 0-100 — confirms ×100 convention

# 1. Eligible pool for this lane: visible + old enough to judge
eligible = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)]
print(f"Eligible pages: {len(eligible):,} of {len(df):,} ({len(eligible)/len(df):.1%})")

# 2. Declining share within eligible pool — describing the data only,
# NEVER using trend_direction/trend_pct as a feature later (label trap)
declining = eligible[eligible["trend_direction"] == "down"]
print(f"Declining among eligible: {len(declining):,} ({len(declining)/len(eligible):.1%})")

# 3. Position data quality check — avg_position=0 means "no data", must exclude
has_position = eligible[eligible["avg_position"] > 0]
print(f"Eligible pages with valid position data: {len(has_position):,} "
      f"({len(has_position)/len(eligible):.1%})")

# 4. Stakes: how much visibility sits in declining pages
share_impr = declining["impressions_90d"].sum() / eligible["impressions_90d"].sum()
print(f"Share of eligible impressions sitting in declining pages: {share_impr:.1%}")


                ctr  engagement_rate     trend_pct
count  30000.000000     30000.000000  26612.000000
mean       0.510733         2.534520     -4.785969
std        3.279162         8.310096    473.861780
min        0.000000         0.000000   -100.000000
25%        0.000000         0.000000    -62.600000
50%        0.070000         0.000000    -33.500000
75%        0.290000         1.350000      0.000000
max      100.000000       100.000000  44900.000000
Eligible pages: 30,000 of 30,000 (100.0%)
Declining among eligible: 16,262 (54.2%)
Eligible pages with valid position data: 28,795 (96.0%)
Share of eligible impressions sitting in declining pages: 51.3%


In [10]:
print(df["impressions_90d"].min(), df["impressions_90d"].eq(0).sum())
print(df["content_age_days"].min(), (df["content_age_days"] < 90).sum())

1 0
90 0


`Of the 30,000 starter pages, all meet the basic eligibility bar (visible, ≥90 days old) — suggesting this slice was pre-curated. Of those, 54.2% are currently trending down, and declining pages hold 51.3% of all impressions in the pool — meaning over half the visibility we're tracking sits on pages already losing ground. That's a large enough, high-stakes-enough pool to justify a systematic review-priority tool rather than ad hoc manual checks. Note: trend_pct shows extreme outliers (max 44,900%, likely low-volume artifacts) — this framing relies on the more robust trend_direction bucket, not raw percentage magnitude.`

## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

`What this work can say:`

- Which pages show observed signals associated with decline, stagnation, or opportunity — based on measured impressions, clicks, position, CTR, and engagement over a trailing window.
- A ranked, decision-support list: which pages a reviewer with limited time should look at first, given the evidence available today.
- Directional patterns e.g. "pages with low CTR at strong positions tend to also show weak engagement" described as associations, not mechanisms.
- Whether a scoring method beats a transparent baseline rule, measured honestly against a held-out set.

`What this work will never say:`

- That refreshing a page causes recovery or improved traffic this data has no experiment or causal design behind it, only observational history. Correlation between "was refreshed" and "later improved" (if I even had refresh-event data, which I don't in the starter set) would not prove causation.
- Anything about why Google ranks a page where it does, or that any signal here reflects a real ranking factor. I only observe outcomes (position, clicks, impressions), never the algorithm producing them.
- That a flagged page is guaranteed to be a problem, or that an unflagged page is guaranteed fine this is triage for limited human attention, not a verdict.
- Any claim built from trend_direction/trend_pct as if it were ground truth rather than a bucket I chose to describe the data it's a proxy, not the real target I'd build for a stronger capstone (a future-window outcome).

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.